# Pooled harmonized observational analysis (CCM revision)

**Purpose (Reviewer 1, Major 3):** harmonize the shared covariate set across the three
observational cohorts (eICU-CRD, PMAP, MIMIC-IV), pool patients with a dataset indicator,
draw a single stratified 70/30 split, and repeat the HTE assessment
(CausalForestDML → CATE intervals, LRT for CATE×TTM interaction, IPW-weighted
interaction test, GATES) for **both outcomes**. Also produces:

- **Reviewer 1, Major 1** — IPW-weighted interaction test (pooled, and within each dataset's test rows)
- **Reviewer 1, Major 2** — L1 (lasso) S-learner sensitivity analysis with penalized treatment interactions
- **Reviewer 1, minor d** — collinearity filter (|r| > 0.7 with TTM) applied **uniformly**, including eICU-CRD
- **Reviewer 1, minor h** — GATES for the neurologic outcome (pooled level; per-dataset GATES-neuro lives in each dataset's DML notebook)

HYPERION is intentionally excluded (randomized design, non-shockable-only population).

**Exposure definitions** (unchanged from the paper): eICU-CRD `treatment_hypothermia`-derived
`Hypothermia` flag from treatment-table documentation; PMAP sustained <36°C for ≥12 consecutive
hours; MIMIC-IV cooling-device use ≥12 consecutive hours.

Run on the cluster from this folder:
```bash
jupyter nbconvert --to notebook --execute pooledObservationalAnalysis.ipynb \
  --output pooledObservationalAnalysis.ipynb --ExecutePreprocessor.timeout=-1
```
The final cell writes `pooled_analysis_results.csv` with the numbers for the
`[INSERT]` slots in `RESPONSE_TO_REVIEWERS.md`.


In [ ]:
# CONFIG
SEED = 42
TEST_SIZE = 0.30
CORR_THRESHOLD = 0.7   # drop covariates with |corr| > 0.7 with TTM (uniform across datasets)
MIN_POSITIVE = 15      # drop binary covariates with < 15 positive cases (pooled)
N_GATES_GROUPS = 5
PS_CLIP = (0.05, 0.95) # propensity truncation, same as GATES in the paper

EICU_CSV  = '../eICU/eICUPredictorsDiag.csv'
PMAP_CSV  = '../pmap/PMAP_Predictors4.csv'
MIMIC_CSV = '../mimiciv/MIMIC_Predictors1.csv'


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegressionCV
from xgboost import XGBClassifier
from econml.dml import CausalForestDML

np.random.seed(SEED)
pd.set_option('display.width', 160)


## 1. Per-dataset loading

Each loader replicates the preprocessing of the corresponding `*Util.py` **faithfully**
(same filters, same dropped columns, same temperature-column exclusions), except that:

1. both outcomes are kept (`mortality`, `neuro_favorable`) — outcome-specific row filters
   are deferred to analysis time (rows where the neurologic outcome is undefined carry NaN);
2. the `top_correlations.csv` collinearity filter (PMAP/MIMIC) is **not** applied here —
   the |r| > 0.7 filter is applied uniformly on the pooled data below (Reviewer 1, minor d);
3. treatment/outcome columns are renamed to canonical names (`TTM`, `mortality`,
   `neuro_favorable`).

Note: the PMAP util's temperature-column exclusion uses `and` where MIMIC uses `or`;
each is replicated as-is to match the published per-dataset cohorts.


In [ ]:
UNSCORABLE = 'Unable to score due to medication'

def load_eicu(path=EICU_CSV):
    df = pd.read_csv(path)
    f = (df['LastMGCS'] != UNSCORABLE) & (~df['LastMGCS'].isna())
    f = f & (df['FirstMGCSTime'] != df['LastMGCSTime'])
    for c in ['FirstGCS', 'FirstMGCS', 'LastMGCS', 'LastGCS']:
        df.loc[df[c] == UNSCORABLE, c] = np.nan
    df.loc[df['DeathAtDischarge'] == 1, 'LastMGCS'] = 1
    df['gender'] = (df['gender'] == 'Male').astype(int)
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'LastMGCS'].astype(float) == 6).astype(int)
    gcs15 = (df['nurse_first_Motor'] != 6)
    df = df[f & gcs15 & ~df['Hypothermia'].isna()].copy()
    temp_cols = [c for c in df.columns if 'emp' in c]
    drop_cols = ['FirstGCS', 'FirstMGCS', 'LastMGCSTime', 'FirstMGCSTime', 'LastMGCS',
                 'apacheadmissiondx', 'hospitaladmittime24', 'FirstGCSTime', 'LastGCSTime',
                 'LastGCS', 'hospitaldischargestatus', 'LastGCS15', 'hospitaladmitsource',
                 'patientunitstayid']
    df = df.drop(columns=[c for c in set(temp_cols + drop_cols) if c in df.columns])
    df = df.rename(columns={'Hypothermia': 'TTM', 'DeathAtDischarge': 'mortality',
                            'LastMGCSPositive': 'neuro_favorable', 'gender': 'male'})
    df = df.select_dtypes(exclude=['object'])
    df['dataset'] = 'eICU'
    return df

def _load_epic_style(path, temp_rule):
    df = pd.read_csv(path)
    f = (df['first_mGCS_time'] != df['last_mGCS_time'])
    df.loc[df['death_at_disch'] == 1, 'last_mGCS'] = 1
    df.loc[f, 'LastMGCSPositive'] = (df.loc[f, 'last_mGCS'].astype(float) == 6).astype(int)
    gcs15 = (df['first_mGCS'] != 6)
    df = df[gcs15 & ~df['hypothermia'].isna()].copy()
    temp_cols = [c for c in df.columns if temp_rule(c)]
    drop_cols = ['first_mGCS', 'last_mGCS_time', 'first_mGCS_time', 'last_mGCS']
    df = df.drop(columns=[c for c in set(temp_cols + drop_cols) if c in df.columns])
    df = df.rename(columns={'hypothermia': 'TTM', 'death_at_disch': 'mortality',
                            'LastMGCSPositive': 'neuro_favorable'})
    df = df.select_dtypes(exclude=['object'])
    return df

def load_pmap(path=PMAP_CSV):
    # PMAP util: 'emp' in x AND x startswith 'dx_' (replicated as-is)
    df = _load_epic_style(path, lambda c: ('emp' in c) and (c[0:3] == 'dx_'))
    df['dataset'] = 'PMAP'
    return df

def load_mimic(path=MIMIC_CSV):
    # MIMIC util: 'emp' in x OR x startswith 'dx_'
    df = _load_epic_style(path, lambda c: ('emp' in c) or (c[0:3] == 'dx_'))
    df['dataset'] = 'MIMIC-IV'
    return df

frames = {'eICU': load_eicu(), 'PMAP': load_pmap(), 'MIMIC-IV': load_mimic()}
for name, d in frames.items():
    print(f"{name:9s} n={len(d):5d}  TTM={int(d['TTM'].sum()):4d} ({d['TTM'].mean():.1%})  "
          f"mortality={d['mortality'].mean():.1%}  neuro defined={d['neuro_favorable'].notna().mean():.1%}  "
          f"raw covariates={d.shape[1]-4}")


## 2. Harmonization

Column names are normalized (lowercase, non-alphanumeric → `_`), a manual rename map lifts
known synonyms onto canonical names, and the **intersection** of covariates across the three
datasets defines the pooled feature set.

**After the first executed run, review the diagnostic output below** (lab convention:
read executed output before finalizing): the "present in exactly 2 datasets" list is where
extending `RENAME_MAPS` buys back features. Extend the map and re-run.


In [ ]:
RESERVED = ['TTM', 'mortality', 'neuro_favorable', 'dataset']

def normalize(c):
    out = ''.join(ch if ch.isalnum() else '_' for ch in c.strip().lower())
    while '__' in out:
        out = out.replace('__', '_')
    return out.strip('_')

# Manual synonym map applied AFTER normalization: {dataset: {normalized_name: canonical}}.
# EXTEND THIS after reviewing the diagnostics printed below.
RENAME_MAPS = {
    'eICU':     {'male': 'male', 'age': 'age', 'nurse_first_motor': 'first_mgcs'},
    'PMAP':     {'sex': 'male', 'gender': 'male', 'age': 'age'},
    'MIMIC-IV': {'sex': 'male', 'gender': 'male', 'age': 'age'},
}

harmonized = {}
for name, d in frames.items():
    d = d.copy()
    ren = {c: normalize(c) for c in d.columns if c not in RESERVED}
    d = d.rename(columns=ren)
    d = d.rename(columns=RENAME_MAPS.get(name, {}))
    d = d.loc[:, ~d.columns.duplicated()]
    harmonized[name] = d

cov_sets = {n: set(d.columns) - set(RESERVED) for n, d in harmonized.items()}
shared = sorted(set.intersection(*cov_sets.values()))
print(f"Shared covariates across all 3 datasets: {len(shared)}")
print(shared)

from itertools import combinations
all_cols = set.union(*cov_sets.values())
in_two = sorted(c for c in all_cols
                if sum(c in s for s in cov_sets.values()) == 2)
print(f"\nPresent in exactly 2 of 3 datasets ({len(in_two)}) — candidates for RENAME_MAPS:")
print(in_two)


## 3. Pooling and uniform feature filters

Pool on the shared covariates, add dataset indicator dummies (site-effect absorption),
then apply — uniformly, including eICU-CRD —

- rare-binary filter: binary covariates with < 15 positive cases in the pool;
- positivity/collinearity filter: covariates with |r| > 0.7 with `TTM`.


In [ ]:
pooled = pd.concat(
    [d[RESERVED + shared] for d in harmonized.values()], axis=0, ignore_index=True)

ds_dummies = pd.get_dummies(pooled['dataset'], prefix='ds', drop_first=True).astype(int)
pooled = pd.concat([pooled, ds_dummies], axis=1)
DS_COLS = list(ds_dummies.columns)

# rare-binary filter
binary = [c for c in shared if pooled[c].dropna().nunique() == 2]
rare = [c for c in binary if pooled[c].sum() < MIN_POSITIVE]

# uniform treatment-collinearity filter (Reviewer 1, minor d)
corr_t = pooled[shared].corrwith(pooled['TTM']).abs()
collinear = [c for c in shared if corr_t.get(c, 0) > CORR_THRESHOLD]

COVARIATES = [c for c in shared if c not in set(rare) | set(collinear)]
print(f"Pooled n = {len(pooled)}  (TTM {pooled['TTM'].mean():.1%})")
print(f"Shared covariates {len(shared)} -> dropped {len(rare)} rare-binary, "
      f"{len(collinear)} collinear with TTM -> {len(COVARIATES)} + {len(DS_COLS)} dataset dummies")
if collinear:
    print('Collinear with TTM (|r|>0.7):', collinear)


## 4. Analysis machinery

Split → preprocess (fit on train only) → CausalForestDML (paper's nuisance settings) →
CATE intervals → LRT (manuscript form: `logit(Y) ~ T + CATE + T×CATE`) → IPW-weighted
interaction test → GATES with IPW.

The weighted-test and GATES functions here are self-contained; the same two functions can
be pasted into each per-dataset `*AnalysisDML.ipynb` to produce the per-dataset weighted
LRT promised in the response letter (Reviewer 1, Major 1).


In [ ]:
def split_and_preprocess(outcome):
    d = pooled.dropna(subset=[outcome]).copy()
    X = d[COVARIATES + DS_COLS]
    T = d['TTM'].astype(int)
    y = d[outcome].astype(int)
    strat = pd.concat([y, T, d['dataset']], axis=1)
    X_tr, X_te, T_tr, T_te, y_tr, y_te, ds_tr, ds_te = train_test_split(
        X, T, y, d['dataset'], test_size=TEST_SIZE, stratify=strat, random_state=SEED)

    num = [c for c in X.columns if X[c].dropna().nunique() > 2]
    scaler = StandardScaler().fit(X_tr[num])
    imputer = KNNImputer(n_neighbors=10)

    def transform(Xp, fit=False):
        Xp = Xp.copy()
        Xp[num] = scaler.transform(Xp[num])
        arr = imputer.fit_transform(Xp) if fit else imputer.transform(Xp)
        return pd.DataFrame(arr, columns=Xp.columns, index=Xp.index)

    return (transform(X_tr, fit=True), transform(X_te),
            T_tr, T_te, y_tr, y_te, ds_tr, ds_te)


def fit_causal_forest(X_tr, T_tr, y_tr):
    cf = CausalForestDML(
        model_y=XGBClassifier(max_depth=3, n_estimators=50),
        model_t=XGBClassifier(max_depth=2, n_estimators=20),
        discrete_treatment=True, discrete_outcome=True,
        random_state=SEED, n_jobs=-1)
    cf.fit(y_tr, T_tr, X=X_tr, cache_values=True)
    return cf


def cf_propensity(cf, X):
    preds = []
    for mc in cf.models_t:
        for mdl in mc:
            p = mdl.predict_proba(np.asarray(X))
            preds.append(p[:, 1] if p.ndim == 2 else np.ravel(p))
    return np.clip(np.mean(np.vstack(preds), axis=0), 1e-6, 1 - 1e-6)


In [ ]:
def lrt_cate_interaction(y, T, cate):
    """Manuscript LRT: logit(Y) ~ T + CATE vs + T:CATE (chi2, 1 df)."""
    df = pd.DataFrame({'const': 1.0, 'T': np.asarray(T, float),
                       'cate': np.asarray(cate, float)})
    df['tx'] = df['T'] * df['cate']
    y = np.asarray(y, float)
    m0 = sm.Logit(y, df[['const', 'T', 'cate']]).fit(disp=False)
    m1 = sm.Logit(y, df[['const', 'T', 'cate', 'tx']]).fit(disp=False)
    lr = 2 * (m1.llf - m0.llf)
    return {'lr_stat': lr, 'p': chi2.sf(lr, 1)}


def ipw_interaction_test(y, T, cate, ps, clip=PS_CLIP):
    """IPW-weighted CATE x TTM interaction (Reviewer 1, Major 1).

    Weighted logistic GLM with stabilized-free IPW as var_weights; reports the Wald
    p for the interaction and the weighted deviance-difference (working LRT)."""
    T = np.asarray(T, float); y = np.asarray(y, float)
    ps = np.clip(np.asarray(ps, float), *clip)
    w = np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    df = pd.DataFrame({'const': 1.0, 'T': T, 'cate': np.asarray(cate, float)})
    df['tx'] = df['T'] * df['cate']
    fam = sm.families.Binomial()
    m0 = sm.GLM(y, df[['const', 'T', 'cate']], family=fam, var_weights=w).fit()
    m1 = sm.GLM(y, df[['const', 'T', 'cate', 'tx']], family=fam, var_weights=w).fit()
    dev = m0.deviance - m1.deviance
    return {'wald_p': m1.pvalues['tx'], 'weighted_lr_stat': dev,
            'weighted_lr_p': chi2.sf(dev, 1)}


def run_gates(cate, y, T, ps, n_groups=N_GATES_GROUPS, title=''):
    """IPW-weighted GATES with joint F-test and Spearman monotonicity (paper method)."""
    ps = np.clip(np.asarray(ps, float), *PS_CLIP)
    d = pd.DataFrame({'y': np.asarray(y, float), 'T': np.asarray(T, float),
                      'cate': np.asarray(cate, float)})
    d['w'] = np.where(d['T'] == 1, 1 / ps, 1 / (1 - ps))
    d['g'] = pd.qcut(d['cate'], q=n_groups, labels=False, duplicates='drop') + 1
    rows = []
    for g, sub in d.groupby('g'):
        wls = sm.WLS(sub['y'], sm.add_constant(sub['T']), weights=sub['w']).fit()
        ci = wls.conf_int()
        rows.append({'group': int(g), 'n': len(sub), 'mean_cate': sub['cate'].mean(),
                     'gate': wls.params.iloc[1],
                     'ci_low': np.asarray(ci)[1, 0], 'ci_high': np.asarray(ci)[1, 1]})
    gdf = pd.DataFrame(rows)

    gd = pd.get_dummies(d['g'].astype(int), prefix='g', drop_first=True).astype(float)
    Xr = sm.add_constant(pd.concat([d[['T']], gd], axis=1))
    Xf = Xr.copy()
    for c in gd.columns:
        Xf[f'T_x_{c}'] = d['T'].values * gd[c].values
    mf = sm.WLS(d['y'], Xf, weights=d['w']).fit()
    mr = sm.WLS(d['y'], Xr, weights=d['w']).fit()
    f_stat, f_p, _ = mf.compare_f_test(mr)
    rho, rho_p = stats.spearmanr(gdf['group'], gdf['gate'])

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(gdf['group'], gdf['gate'],
                yerr=[gdf['gate'] - gdf['ci_low'], gdf['ci_high'] - gdf['gate']],
                fmt='o-', capsize=5)
    ax.axhline(0, ls='--', color='gray')
    ax.set_xlabel('CATE quintile (1 = lowest predicted benefit)')
    ax.set_ylabel('GATE (IPW)')
    ax.set_title(f'GATES — {title}\nF p={f_p:.3f}, Spearman rho={rho:.2f} (p={rho_p:.3f})')
    plt.tight_layout()
    plt.savefig(f"gates_pooled_{title.lower().replace(' ', '_').replace('=', '')}.png", dpi=150)
    plt.show()
    return gdf, {'f_stat': f_stat, 'f_p': f_p, 'spearman_rho': rho, 'spearman_p': rho_p}


## 5. Pooled analysis — both outcomes

For each outcome: fit CausalForestDML on pooled train, evaluate on the pooled held-out
test set. Per-dataset weighted interaction tests use the pooled model's CATE and
propensity restricted to each dataset's test rows (complementary to — not a substitute
for — the per-dataset models in the dataset notebooks).


In [ ]:
RESULTS = []

def run_pooled(outcome, label):
    X_tr, X_te, T_tr, T_te, y_tr, y_te, ds_tr, ds_te = split_and_preprocess(outcome)
    print(f"=== {label}: train n={len(X_tr)}, test n={len(X_te)} ===")
    cf = fit_causal_forest(X_tr, T_tr, y_tr)

    cate_te = np.ravel(cf.effect(X_te))
    lo, hi = cf.effect_interval(X_te, alpha=0.05)
    lo, hi = np.ravel(lo), np.ravel(hi)
    pct_ci_cross_zero = float(np.mean((lo < 0) & (hi > 0)))

    order = np.argsort(cate_te)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(cate_te[order], label='CATE')
    ax.fill_between(range(len(order)), lo[order], hi[order], alpha=0.2, label='95% CI')
    ax.axhline(0, color='red', ls='--', lw=1)
    ax.set_xlabel('Patient (sorted)'); ax.set_ylabel('CATE')
    ax.set_title(f'Pooled observational CATE — {label}')
    ax.legend(); plt.tight_layout()
    plt.savefig(f"cate_pooled_{label.lower().replace(' ', '_')}.png", dpi=150)
    plt.show()

    ps_te = cf_propensity(cf, X_te)
    lrt = lrt_cate_interaction(y_te, T_te, cate_te)
    ipw = ipw_interaction_test(y_te, T_te, cate_te, ps_te)
    gdf, gates = run_gates(cate_te, y_te, T_te, ps_te, title=label)

    print(f"LRT (unweighted)      : chi2={lrt['lr_stat']:.3f}, p={lrt['p']:.4f}")
    print(f"IPW interaction (Wald): p={ipw['wald_p']:.4f}; "
          f"weighted-LR p={ipw['weighted_lr_p']:.4f}")
    print(f"GATES: F p={gates['f_p']:.4f}, Spearman rho={gates['spearman_rho']:.2f} "
          f"(p={gates['spearman_p']:.4f}); CATE CIs crossing zero: {pct_ci_cross_zero:.1%}")
    display(gdf.round(3))

    RESULTS.append({'analysis': 'pooled', 'outcome': label,
                    'n_train': len(X_tr), 'n_test': len(X_te),
                    'lrt_p': lrt['p'], 'ipw_wald_p': ipw['wald_p'],
                    'ipw_lr_p': ipw['weighted_lr_p'], 'gates_f_p': gates['f_p'],
                    'gates_spearman_rho': gates['spearman_rho'],
                    'gates_spearman_p': gates['spearman_p'],
                    'pct_cate_ci_cross_zero': pct_ci_cross_zero})

    # per-dataset weighted interaction on the pooled model (test rows only)
    for ds_name in ds_te.unique():
        m = (ds_te == ds_name).values
        try:
            r = ipw_interaction_test(np.asarray(y_te)[m], np.asarray(T_te)[m],
                                     cate_te[m], ps_te[m])
            print(f"  {ds_name:9s} (test n={m.sum():4d}) IPW interaction Wald p={r['wald_p']:.4f}")
            RESULTS.append({'analysis': f'per-dataset ({ds_name})', 'outcome': label,
                            'n_test': int(m.sum()), 'ipw_wald_p': r['wald_p'],
                            'ipw_lr_p': r['weighted_lr_p']})
        except Exception as e:
            print(f"  {ds_name}: weighted test failed ({e})")
    return cf

cf_mort = run_pooled('mortality', 'Hospital mortality')


In [ ]:
cf_neuro = run_pooled('neuro_favorable', 'Favorable neurologic (last mGCS=6)')

## 6. Lasso S-learner sensitivity (Reviewer 1, Major 2)

L1-penalized logistic S-learner on the pooled training set with design
`[X, T, X×T]` — every treatment interaction penalized. `C` chosen by 5-fold CV.
Reported: how many interaction coefficients survive the penalty (0 ⇒ no heterogeneity
signal survives regularization).


In [ ]:
def lasso_slearner(outcome, label):
    X_tr, X_te, T_tr, T_te, y_tr, y_te, *_ = split_and_preprocess(outcome)
    Xd = X_tr.copy()
    Xd['TTM'] = np.asarray(T_tr, float)
    inter_cols = []
    for c in X_tr.columns:
        ic = f'{c}_x_TTM'
        Xd[ic] = Xd[c] * Xd['TTM']
        inter_cols.append(ic)
    clf = LogisticRegressionCV(penalty='l1', solver='saga', Cs=10, cv=5,
                               scoring='neg_log_loss', max_iter=5000,
                               n_jobs=-1, random_state=SEED)
    clf.fit(Xd, y_tr)
    coefs = pd.Series(clf.coef_[0], index=Xd.columns)
    nz = coefs[inter_cols][coefs[inter_cols] != 0]
    print(f"{label}: C={clf.C_[0]:.4g}; non-zero interaction coefficients: "
          f"{len(nz)}/{len(inter_cols)}")
    if len(nz):
        display(nz.sort_values(key=np.abs, ascending=False).head(15).round(4))
    RESULTS.append({'analysis': 'lasso S-learner', 'outcome': label,
                    'n_train': len(Xd), 'nonzero_interactions': len(nz),
                    'n_interactions': len(inter_cols), 'chosen_C': float(clf.C_[0])})
    return coefs

_ = lasso_slearner('mortality', 'Hospital mortality')
_ = lasso_slearner('neuro_favorable', 'Favorable neurologic (last mGCS=6)')


## 7. Summary for the response letter

Everything below maps to `[INSERT]` slots in `RESPONSE_TO_REVIEWERS.md`
(R1 Major 1, 2, 3; minor d, h).


In [ ]:
summary = pd.DataFrame(RESULTS)
summary.to_csv('pooled_analysis_results.csv', index=False)
display(summary.round(4))

pool_rows = summary[summary['analysis'] == 'pooled']
print('\n--- Paste-ready for the response letter ---')
print(f"Pooled harmonized observational cohort: n = {len(pooled)} "
      f"({int(pooled['TTM'].sum())} TTM, {pooled['TTM'].mean():.1%}); "
      f"{len(COVARIATES)} shared covariates + {len(DS_COLS)} dataset indicators.")
for _, r in pool_rows.iterrows():
    print(f"[{r['outcome']}] interaction LRT p = {r['lrt_p']:.3f}; "
          f"IPW-weighted interaction p = {r['ipw_wald_p']:.3f}; "
          f"GATES F-test p = {r['gates_f_p']:.3f}; "
          f"Spearman rho = {r['gates_spearman_rho']:.2f} (p = {r['gates_spearman_p']:.3f}); "
          f"{r['pct_cate_ci_cross_zero']:.1%} of CATE 95% CIs cross zero.")
